# Clustering Deep Dive — 5‑minute demo notebook

Goal: a compact, runnable Jupyter notebook that demonstrates clustering concepts, shows step‑by‑step what happens under the hood, and includes short demos for K‑Means (from scratch + sklearn), Gaussian Mixture Model (EM intuition + sklearn), Hierarchical clustering (dendrogram), and DBSCAN. Each section contains short explanations and visualizations you can run during a 5‑minute presentation.

Run cells top to bottom. Keep the presentation pace: show concept, run the small demo cell, highlight the plot, move on.

## 1. Imports and helper functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage
from matplotlib.patches import Ellipse

sns.set(style='whitegrid')

def plot_2d(X, labels=None, centers=None, title=None, ax=None, cmap='tab10'):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5,4))
    if labels is None:
        ax.scatter(X[:,0], X[:,1], s=30)
    else:
        ax.scatter(X[:,0], X[:,1], c=labels, s=30, cmap=cmap)
    if centers is not None:
        ax.scatter(centers[:,0], centers[:,1], c='black', s=100, marker='x')
    if title:
        ax.set_title(title)
    return ax

def draw_ellipse(position, covariance, ax=None, kwargs):
    """Draw an ellipse with a given position and covariance"""
    ax = ax or plt.gca()
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height,
                             angle, alpha=0.2, kwargs))
    return ax


## 2. Synthetic datasets (quickly show different cluster shapes)

We create three datasets: Gaussian blobs (well separated), moons (non‑convex), and concentric circles (non‑convex, nested). Use these to show algorithm strengths/weaknesses.

In [ ]:
rng = np.random.RandomState(42)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.60, random_state=rng)
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=rng)
X_circles, y_circles = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=rng)

fig, axes = plt.subplots(1,3, figsize=(15,4))
plot_2d(X_blobs, y_blobs, title='Gaussian blobs', ax=axes[0])
plot_2d(X_moons, y_moons, title='Two moons', ax=axes[1])
plot_2d(X_circles, y_circles, title='Concentric circles', ax=axes[2])
plt.tight_layout()


## 3. K‑Means intuition and from‑scratch demo (step by step)

Concepts:
- K‑Means partitions points by assigning each point to the nearest centroid (Voronoi regions).
- Iterative steps: (1) assign points to nearest centroid, (2) recompute centroids as cluster means, (3) repeat until convergence.
- Objective: minimize within‑cluster sum of squared distances (inertia).
 
Below: a compact from‑scratch implementation that records intermediate centroids and assignments so we can plot the algorithm's progress.

In [ ]:
def kmeans_scratch(X, k, max_iters=10, tol=1e-4, random_state=0):
    rng = np.random.RandomState(random_state)
    # initialize centroids by sampling k points
    centers = X[rng.choice(len(X), k, replace=False)].astype(float)
    history = [(centers.copy(), None)]
    for it in range(max_iters):
        # assignment step
        dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
        labels = np.argmin(dists, axis=1)
        # update step
        new_centers = np.array([X[labels == j].mean(axis=0) if np.any(labels==j) else centers[j]
                                for j in range(k)])
        history.append((new_centers.copy(), labels.copy()))
        shift = np.linalg.norm(new_centers - centers)
        centers = new_centers
        if shift < tol:
            break
    return centers, labels, history

# Run on blobs and plot iterations
centers, labels, history = kmeans_scratch(X_blobs, k=4, max_iters=10, random_state=1)

fig, axes = plt.subplots(2, 3, figsize=(15,8))
axes = axes.ravel()
for i, (c, lbls) in enumerate(history[:6]):
    ax = axes[i]
    if lbls is None:
        ax.scatter(X_blobs[:,0], X_blobs[:,1], s=30)
        ax.scatter(c[:,0], c[:,1], c='black', marker='x', s=100)
        ax.set_title(f'Init centers')
    else:
        ax.scatter(X_blobs[:,0], X_blobs[:,1], c=lbls, s=30, cmap='tab10')
        ax.scatter(c[:,0], c[:,1], c='black', marker='x', s=100)
        ax.set_title(f'Iteration {i}')
plt.tight_layout()


### Talking points for K‑Means demo
- Show the initial random centroids, then the assignment step (Voronoi regions) and centroid updates.
- Explain why K‑Means fails on non‑convex shapes (moons, circles): it assumes spherical clusters and uses Euclidean distance to means.
- Mention sensitivity to initialization and the elbow method / silhouette score for choosing k.

In [ ]:
# Quick comparison: KMeans on different shapes
fig, axes = plt.subplots(1,3, figsize=(15,4))
km = KMeans(n_clusters=2, random_state=0)
labels_moons = km.fit_predict(X_moons)
plot_2d(X_moons, labels_moons, title='KMeans on moons (bad)', ax=axes[0])
labels_circles = km.fit_predict(X_circles)
plot_2d(X_circles, labels_circles, title='KMeans on circles (bad)', ax=axes[1])
km4 = KMeans(n_clusters=4, random_state=0)
labels_blobs = km4.fit_predict(X_blobs)
plot_2d(X_blobs, labels_blobs, title='KMeans on blobs (good)', ax=axes[2])
plt.tight_layout()


## 4. Model selection: Elbow and Silhouette

Quick code to compute inertia (KMeans objective) and silhouette score for a range of k. Use this to justify a choice of k during the talk.

In [ ]:
ks = range(2,8)
inertias = []
silhs = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=0).fit(X_blobs)
    inertias.append(km.inertia_)
    silhs.append(silhouette_score(X_blobs, km.labels_))

fig, ax1 = plt.subplots(figsize=(6,4))
ax1.plot(ks, inertias, '-o', label='inertia')
ax1.set_xlabel('k')
ax1.set_ylabel('inertia')
ax2 = ax1.twinx()
ax2.plot(ks, silhs, '-s', color='C1', label='silhouette')
ax2.set_ylabel('silhouette score')
ax1.set_title('Elbow (inertia) and silhouette for blobs')
ax1.grid(True)
fig.tight_layout()


## 5. Gaussian Mixture Model (EM) — intuition and visualization of responsibilities

Concepts:
- GMM models data as a mixture of Gaussians. Each component has a mean, covariance, and mixing weight.
- EM alternates between: E‑step (compute responsibilities = soft assignments) and M‑step (update parameters weighted by responsibilities).
- GMM can model elliptical clusters and overlapping clusters; it provides soft cluster membership probabilities.
Below: fit a GMM and visualize component ellipses and responsibilities for a few points.

In [ ]:
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=0).fit(X_blobs)
labels_gmm = gmm.predict(X_blobs)
probs = gmm.predict_proba(X_blobs)

fig, ax = plt.subplots(figsize=(6,5))
plot_2d(X_blobs, labels_gmm, title='GMM clustering (soft assignments)', ax=ax)
centers = gmm.means_
for i, (mean, cov) in enumerate(zip(gmm.means_, gmm.covariances_)):
    draw_ellipse(mean, cov, ax=ax, edgecolor=f'C{i}')
ax.scatter(centers[:,0], centers[:,1], c='black', s=80, marker='x')
plt.tight_layout()

# Show responsibilities for a few example points
idx = np.random.choice(len(X_blobs), 6, replace=False)
fig, axes = plt.subplots(2,3, figsize=(12,6))
for i, j in enumerate(idx):
    ax = axes.ravel()[i]
    ax.scatter(X_blobs[:,0], X_blobs[:,1], c='lightgray', s=20)
    ax.scatter(X_blobs[j,0], X_blobs[j,1], c='red', s=80)
    # show arrows to component means weighted by responsibility
    for k in range(gmm.n_components):
        r = probs[j, k]
        ax.arrow(X_blobs[j,0], X_blobs[j,1],
                 (gmm.means_[k,0]-X_blobs[j,0])r,
                 (gmm.means_[k,1]-X_blobs[j,1])r,
                 head_width=0.05, length_includes_head=True, color=f'C{k}', alpha=0.8)
    ax.set_title(f'Responsibilities: {np.round(probs[j],2)}')
plt.tight_layout()


## 6. Hierarchical clustering and dendrogram

Show linkage and dendrogram. Useful to explain agglomerative merging and how to choose number of clusters by cutting the tree.

In [ ]:
Z = linkage(X_blobs, method='ward')
plt.figure(figsize=(8,4))
dendrogram(Z, truncate_mode='lastp', p=12, leaf_rotation=45., leaf_font_size=12.)
plt.title('Hierarchical clustering dendrogram (truncated)')
plt.xlabel('Cluster index')
plt.ylabel('Distance')
plt.tight_layout()


## 7. DBSCAN — density based clustering (core, border, noise)

DBSCAN groups points that are closely packed together, marking points in low‑density regions as noise. It can find arbitrarily shaped clusters and does not require k.
We demonstrate DBSCAN on moons and circles.

In [ ]:
db_moons = DBSCAN(eps=0.2, min_samples=5).fit(X_moons)
labels_db_moons = db_moons.labels_
db_circles = DBSCAN(eps=0.15, min_samples=5).fit(X_circles)
labels_db_circles = db_circles.labels_

fig, axes = plt.subplots(1,2, figsize=(12,5))
plot_2d(X_moons, labels_db_moons, title='DBSCAN on moons', ax=axes[0])
plot_2d(X_circles, labels_db_circles, title='DBSCAN on circles', ax=axes[1])
plt.tight_layout()

# Show how noise is labeled as -1
print('Unique labels (moons):', np.unique(labels_db_moons))
print('Unique labels (circles):', np.unique(labels_db_circles))


## 8. Quick comparison table and closing notes

Use this slide to summarize strengths/weaknesses and practical tips.

| Algorithm | Strengths | Weaknesses | When to use |
|---|---|---|---|
| K‑Means | Fast; simple; good for spherical clusters | Assumes convex/spherical clusters; needs k; sensitive to init | Large datasets with roughly spherical clusters |
| GMM (EM) | Soft assignments; models covariance (elliptical clusters) | Can overfit; needs k; more parameters | Overlapping clusters; need probabilistic membership |
| Hierarchical | Dendrogram gives multi‑scale view; no need to predefine k | O(n^2) memory/time for naive implementations | Small datasets; exploratory analysis |
| DBSCAN | Finds arbitrary shapes; identifies noise; no k | Sensitive to eps/min_samples; struggles with varying density | Non‑convex clusters; noisy data |

Practical tips:
- Always visualize data in 2D/3D when possible.
- Standardize features when scales differ.
- Use multiple algorithms and metrics (silhouette, domain knowledge) to validate clusters.
- For high‑dimensional data, reduce dimensionality (PCA/UMAP/t‑SNE) before clustering for visualization and sometimes for clustering itself.

### End of notebook — short checklist for your 5‑minute talk
- 30s: What is clustering and why it matters.
- 60s: K‑Means intuition + run the from‑scratch demo (show iterations).
- 30s: Show K‑Means failure on moons/circles.
- 45s: GMM intuition + responsibilities visualization.
- 30s: DBSCAN quick demo on moons/circles.
- 45s: Hierarchical dendrogram + model selection (elbow/silhouette) + closing tips.

Good luck — run the notebook once before presenting to ensure plotting backend and package versions are available.